In [1]:
pip install torch torchvision opencv-python pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install python-telegram-bot torch torchvision pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os
import time
import pickle
from tqdm import tqdm  # <-- новая библиотека

# --- 1. Настройка параметров ---
DATA_DIR = './dataset'
NUM_EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.001
IMAGE_SIZE = 224

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

# --- 2. Аугментация и предобработка ---
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 3. Загрузка данных ---
train_dataset = datasets.ImageFolder(root=f'{DATA_DIR}/train', transform=train_transforms)
val_dataset = datasets.ImageFolder(root=f'{DATA_DIR}/val', transform=val_transforms)

class_names = train_dataset.classes
print(f"Найдено классов: {len(class_names)}")
print(f"Классы: {class_names}")

# Сохраняем имена классов
with open('classes.pkl', 'wb') as f:
    pickle.dump(class_names, f)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- 4. Создание модели ---
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))
model = model.to(device)

# --- 5. Настройка обучения ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

best_val_acc = 0.0

print("Начинаем обучение...")
for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # ---------- Обучение ----------
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    # Оборачиваем train_loader в tqdm для отображения прогресса
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)
    for images, labels in train_progress:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

        # Обновляем описание прогресс-бара (показываем текущую потерю и точность)
        current_loss = running_loss / (total_train / BATCH_SIZE) if total_train > 0 else 0
        current_acc = 100 * correct_train / total_train if total_train > 0 else 0
        train_progress.set_postfix({
            'Loss': f'{current_loss:.4f}',
            'Acc': f'{current_acc:.2f}%'
        })

    train_acc = 100 * correct_train / total_train
    avg_train_loss = running_loss / len(train_loader)

    # ---------- Валидация ----------
    model.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0

    # Для валидации тоже сделаем прогресс-бар, но без обновления после каждого батча (можно просто показать)
    val_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]", leave=False)
    with torch.no_grad():
        for images, labels in val_progress:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

            # Можно показывать текущую точность на валидации в процессе
            val_acc_so_far = 100 * correct_val / total_val if total_val > 0 else 0
            val_progress.set_postfix({'Val Acc': f'{val_acc_so_far:.2f}%'})

    val_acc = 100 * correct_val / total_val
    avg_val_loss = val_loss / len(val_loader)

    epoch_time = time.time() - start_time
    print(f'Эпоха [{epoch+1}/{NUM_EPOCHS}] завершена за {epoch_time:.2f}с')
    print(f'  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print('---')

    # Сохраняем лучшую модель
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  ✅ Новая лучшая модель сохранена (Val Acc: {val_acc:.2f}%)')

    scheduler.step(val_acc)

print("Обучение завершено!")
print(f"Лучшая точность на валидации: {best_val_acc:.2f}%")
torch.save(model.state_dict(), 'smeshariki_classifier.pth')

Используется устройство: cpu
Найдено классов: 3
Классы: ['Barash', 'Hedgehog', 'Krosh']
Начинаем обучение...


Эпоха [1/5] завершена за 8.87с
  Train Loss: 1.3036, Train Acc: 25.81%
  Val Loss: 0.8662, Val Acc: 54.55%
---
  ✅ Новая лучшая модель сохранена (Val Acc: 54.55%)


Эпоха [2/5] завершена за 8.29с
  Train Loss: 0.2104, Train Acc: 100.00%
  Val Loss: 0.9390, Val Acc: 63.64%
---
  ✅ Новая лучшая модель сохранена (Val Acc: 63.64%)


Эпоха [3/5] завершена за 8.35с
  Train Loss: 0.2379, Train Acc: 90.32%
  Val Loss: 0.4069, Val Acc: 81.82%
---
  ✅ Новая лучшая модель сохранена (Val Acc: 81.82%)


Эпоха [4/5] завершена за 8.53с
  Train Loss: 0.0706, Train Acc: 100.00%
  Val Loss: 0.8686, Val Acc: 81.82%
---


Эпоха [5/5] завершена за 8.71с
  Train Loss: 0.1681, Train Acc: 93.55%
  Val Loss: 1.8179, Val Acc: 72.73%
---
Обучение завершено!
Лучшая точность на валидации: 81.82%
